# Perluasan Data Training dan Seleksi URL/QR v8

Mengikuti [protokol sebelum fitting](ai-agent-docs/data-expansion-v8-plan.md). Satu keluarga Logistic Regression; data tambahan dari benchmark TR-OP dengan validasi manual yang dilaporkan penulis. Domain di luar exposure v1-v7 dibagi sebelum prediksi; publisher/source yang sama boleh masuk development dan test melalui domain berbeda, sehingga tidak diklaim independent-source atau gold Indonesia.

Jalankan Run All pada environment Python 3.12 ML, termasuk extra research untuk intake. Cache tersanitasi, checksum dan snapshot registry membuat intake dapat dilanjutkan. Model lama dan API shadow dipertahankan. Notebook tidak membuka target atau menampilkan baris URL dataset.

In [1]:
import gc
import hashlib
import json
import sqlite3
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse import csr_matrix, hstack
from scipy.special import logit
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sklearn.preprocessing import normalize
from threadpoolctl import threadpool_limits

from guardian_ml.metadata import checksum, psl_identity
from guardian_ml.paths import ROOT
from guardian_ml.preprocessing import prepare_url
from guardian_ml.rich_artifact import RichArtifact
from guardian_ml.rich_features import numeric_values as old_numeric
from guardian_ml.rich_features import probabilities as old_probabilities
from guardian_ml.structural_features import (
    SCHEMA_VERSION,
    SPECIFICATION,
    StructuralEncoder,
    structural_values,
    text_matrix,
    probabilities,
    raw_probabilities,
)
from guardian_ml.tr_op_source import prepare_source
from guardian_ml.v8_selection import (
    fit_text_encoder,
    guarded_diagnostics,
    choose_candidate,
)
from guardian_ml.structural_evaluation import (
    paired_domain_interval,
    safe_metrics,
    selection_diagnostics,
    source_point,
)

SEED = 20261101
RUN = ROOT / "ml/runs/notebook-tfidf-logistic-v8"
REPORTS, DATA, ARTIFACTS = RUN / "reports", RUN / "data", RUN / "artifacts"
for folder in [REPORTS, DATA, ARTIFACTS]:
    folder.mkdir(parents=True, exist_ok=True)
NOTEBOOK = ROOT / "guardian_lab_url_qr_expansion_v8.ipynb"
MODEL = ARTIFACTS / "url-core-logistic-v8.joblib"
FROZEN = REPORTS / "frozen-v8.json"
REUSE_FROZEN = FROZEN.exists()
V6 = ROOT / "ml/runs/notebook-tfidf-logistic-v6"
V5 = ROOT / "ml/runs/notebook-tfidf-logistic-v5"
LOCAL = ROOT / "ml/data/expansion-v8"
LOCAL.mkdir(parents=True, exist_ok=True)
base_artifact = RichArtifact()
base = base_artifact.bundle
protected = {
    path: checksum(ROOT / path)
    for path in [
        "guardian_lab_url_qr.ipynb",
        "guardian_lab_url_qr_data_v6.ipynb",
        "ml/runs/notebook-tfidf-logistic-v6/artifacts/url-core-logistic-v6.joblib",
        "ml/runs/notebook-tfidf-logistic-v6/reports/frozen-v6.json",
        "ml/policies/shadow-runtime-manifest.json",
    ]
}
V7 = ROOT / "ml/runs/notebook-tfidf-logistic-v7"
v7_frozen = json.loads((V7 / "reports/frozen-v7.json").read_text())
assert (
    checksum(V7 / "artifacts/url-core-logistic-v7.joblib") == v7_frozen["bundle_sha256"]
)
v7_bundle = joblib.load(V7 / "artifacts/url-core-logistic-v7.joblib")
for path in [
    "guardian_lab_url_qr_structural_v7.ipynb",
    "ml/runs/notebook-tfidf-logistic-v7/artifacts/url-core-logistic-v7.joblib",
    "ml/runs/notebook-tfidf-logistic-v7/reports/frozen-v7.json",
]:
    protected[path] = checksum(ROOT / path)
CONFIGS = [
    {
        "name": "expanded_rich_c2_sqrt",
        "kind": "repeat_v6",
        "C": 2.0,
        "source_power": 0.5,
    },
    {
        "name": "expanded_structural_c2_sqrt",
        "kind": "linear",
        "C": 2.0,
        "source_power": 0.5,
    },
    {
        "name": "expanded_structural_c2_equal",
        "kind": "linear",
        "C": 2.0,
        "source_power": 1.0,
    },
]


def write_json(path, value):
    Path(path).write_text(
        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
        encoding="utf-8",
    )


def hash_value(value):
    return hashlib.sha256(
        json.dumps(value, sort_keys=True, ensure_ascii=True).encode()
    ).hexdigest()


def notebook_code_hash():
    document = json.loads(NOTEBOOK.read_text(encoding="utf-8"))
    return hashlib.sha256(
        "\n\n".join(
            "".join(c["source"]) if isinstance(c["source"], list) else c["source"]
            for c in document["cells"]
            if c["cell_type"] == "code"
        ).encode()
    ).hexdigest()


def canonical_frame(frame):
    for row in frame[["core", "digest", "group"]].itertuples(index=False):
        parsed = prepare_url(row.core)
        assert (
            parsed.core == row.core
            and parsed.digest == row.digest
            and parsed.group == row.group
        )
    return True


print(
    "Research v8: expand training data, enforce per-source development guard, then test."
)

Research v8: expand training data, enforce per-source development guard, then test.


## Data tambahan dan audit gabungan

Intake hanya membaca teks URL dari info.txt dengan HTTP range bounded serta pemeriksaan CRC. Checkpoint tersanitasi dipertahankan. Domain metadata yang sudah dikenal dikeluarkan secara konservatif; URL aktual diperiksa lagi. Source manifest mengungkap exclusion dan exposure metadata v7.

Partisi domain tambahan 50/10/10/10/20 ditetapkan sebelum fitting. Fit v7 menjadi parent; benchmark Phish360 tetap di luar development. Vocabulary/scaler baru di-fit hanya pada gabungan fit. Seluruh anggota konflik representasi fit dikarantina tanpa relabel.

In [2]:
source_manifest = prepare_source(ROOT)
extra = pd.read_parquet(ROOT / source_manifest["output_path"])
parent_path = V6 / "data/development-v6.parquet"
parent = pd.read_parquet(parent_path)
parent_fit_path = V7 / "data/fit-v7.parquet"
parent_fit = pd.read_parquet(parent_fit_path)
assert checksum(parent_fit_path) == v7_frozen["metadata"]["fit_sha256"]
columns = ["core", "digest", "group", "label", "partition", "training_source", "date"]
development = pd.concat(
    [
        parent.loc[~parent.partition.eq("fit"), columns],
        parent_fit[columns],
        extra.loc[~extra.partition.eq("new_test"), columns],
    ],
    ignore_index=True,
)
fit_before = development[development.partition.eq("fit")].reset_index(drop=True)
view = fit_before.core.str.replace(r"^https?://", "", regex=True)
conflict_views = set(
    fit_before.assign(view=view)
    .groupby("view")
    .label.nunique()
    .loc[lambda x: x > 1]
    .index
)
quarantined = fit_before[view.isin(conflict_views)]
fit = fit_before[~view.isin(conflict_views)].reset_index(drop=True)
development = pd.concat(
    [development[~development.partition.eq("fit")], fit], ignore_index=True
)
assert development.digest.is_unique and canonical_frame(development)
partitions = ["fit", "tune", "calibration", "threshold"]
for i, left in enumerate(partitions):
    groups = set(development.loc[development.partition.eq(left), "group"])
    for right in partitions[i + 1 :]:
        assert groups.isdisjoint(
            development.loc[development.partition.eq(right), "group"]
        )
    assert (
        development[development.partition.eq(left)]
        .groupby("training_source")
        .label.nunique()
        .eq(2)
        .all()
    )
tune = development[development.partition.eq("tune")].reset_index(drop=True)
calibration = development[development.partition.eq("calibration")].reset_index(
    drop=True
)
threshold_data = development[development.partition.eq("threshold")].reset_index(
    drop=True
)
external = extra[extra.partition.eq("new_test")].reset_index(drop=True)
assert set(external.label.unique()) == {0, 1}
assert set(external.group).isdisjoint(development.group)
assert set(external.digest).isdisjoint(development.digest)
assert canonical_frame(external)
snapshot = LOCAL / "exposure-before-v8.sqlite"
assert checksum(snapshot) == source_manifest["registry_sha256"]
with sqlite3.connect("file:" + snapshot.as_posix() + "?mode=ro", uri=True) as db:
    for row in extra.itertuples(index=False):
        assert not db.execute(
            "SELECT 1 FROM url_hashes WHERE value=?", (row.digest,)
        ).fetchone()
        assert not db.execute(
            "SELECT 1 FROM domain_hashes WHERE value=?",
            (hashlib.sha256(row.group.encode()).hexdigest(),),
        ).fetchone()
checkpoint = {}
for line in (LOCAL / "sanitized-intake.jsonl").read_text(encoding="utf-8").splitlines():
    item = json.loads(line)
    if item["status"] == "valid":
        checkpoint[item["id"]] = item["record"]
exposure = pd.DataFrame(checkpoint.values())[columns]
outputs = {}
for name, frame, filename in [
    ("development", development, "development-v8.parquet"),
    ("new_tr_op_test", external, "tr-op-test-v8.parquet"),
    ("intake_exposure", exposure, "intake-exposure-v8.parquet"),
]:
    path = DATA / filename
    if not path.exists():
        frame.to_parquet(path, index=False)
    else:
        assert frame.equals(pd.read_parquet(path))
    outputs[name] = {"filename": filename, "sha256": checksum(path), "rows": len(frame)}
fit_audit = {
    "parent_v6_development_sha256": checksum(parent_path),
    "parent_v7_fit_sha256": checksum(parent_fit_path),
    "parent_fit_rows": len(parent_fit),
    "extra_fit_rows": int(extra.partition.eq("fit").sum()),
    "fit_rows": len(fit),
    "new_fit_quarantine_rows": len(quarantined),
    "development_sha256": outputs["development"]["sha256"],
    "source_manifest_sha256": checksum(LOCAL / "tr-op-manifest.json"),
    "source_partition_counts": extra.groupby(["partition", "label"])
    .size()
    .rename("rows")
    .reset_index()
    .to_dict("records"),
    "development_source_class_counts": development.groupby(
        ["partition", "training_source", "label"]
    )
    .size()
    .rename("rows")
    .reset_index()
    .to_dict("records"),
    "automatic_relabels_applied": 0,
    "campaign_independence": "unverified",
}
test_manifest = {
    "created_at_utc": source_manifest["created_at_utc"],
    "source_provenance": source_manifest,
    "outputs": outputs,
    "holdout_scope": "new_domains_same_publisher_source_as_added_development_not_temporal_or_indonesian_gold",
    "new_test_predictions_observed_at_freeze": False,
}
write_json(REPORTS / "development-audit.json", fit_audit)
write_json(REPORTS / "frozen-data-test-manifest.json", test_manifest)
print(
    "Fit rows:",
    len(fit),
    "; newly added fit:",
    fit_audit["extra_fit_rows"],
    "; fresh test:",
    len(external),
)
display(pd.DataFrame(source_manifest["partition_class_counts"]))

Fit rows: 535504 ; newly added fit: 2753 ; fresh test: 1097


,partition,label,rows
0,calibration,0,180
1,calibration,1,351
2,fit,0,1005
3,fit,1,1748
4,new_test,0,384
5,new_test,1,713
6,threshold,0,182
7,threshold,1,330
8,tune,0,199
9,tune,1,346


## Kode, data dan fitur dibekukan sebelum fitting

Tiga konfigurasi ditetapkan sebelum training. Seleksi mengutamakan sumber terlemah dan guard regresi 1 percentage point per sumber terhadap frozen v6, pada aturan threshold development yang sama. Guard ini screening empiris, bukan pengujian non-inferiority statistik.

In [3]:
code_hash = notebook_code_hash()
source_paths = [
    "ml/src/guardian_ml/v8_selection.py",
    "ml/src/guardian_ml/tr_op_source.py",
    "ml/src/guardian_ml/zip_ranges.py",
    "ml/src/guardian_ml/structural_features.py",
    "ml/src/guardian_ml/structural_evaluation.py",
    "ml/src/guardian_ml/rich_features.py",
    "ml/src/guardian_ml/features.py",
    "ml/src/guardian_ml/preprocessing.py",
    "ml/src/guardian_ml/evaluation.py",
]
source_hashes = {name: checksum(ROOT / name) for name in source_paths}
specification = {
    "configs": CONFIGS,
    "feature_specification": SPECIFICATION,
    "data_audit": fit_audit,
    "data_test_manifest_sha256": checksum(REPORTS / "frozen-data-test-manifest.json"),
    "code_sha256": code_hash,
    "source_hashes": source_hashes,
    "seed": SEED,
    "solver": "liblinear",
    "max_iter": 1500,
    "tol": 0.0001,
    "guard_margin": 0.01,
    "selection": "guard_each_source_then_min_recall_macro_recall_AP",
    "versions": base_artifact.metadata["versions"],
    "psl": psl_identity(),
    "vocabulary": {
        "source_cap": 60000,
        "total_cap": 150000,
        "char_max": 100000,
        "word_max": 20000,
        "fit_only": True,
    },
}
spec_hash = hash_value(specification)
if REUSE_FROZEN:
    frozen = json.loads(FROZEN.read_text())
    assert (
        frozen["spec_sha256"] == spec_hash
        and checksum(MODEL) == frozen["bundle_sha256"]
    )
    bundle = joblib.load(MODEL)
    validation_results = json.loads((REPORTS / "validation-results.json").read_text())
    selected = frozen["selected"]
    selection_decision = frozen["selection_decision"]
else:
    assert not MODEL.exists() and not (REPORTS / "evaluation-v8.json").exists()
    write_json(
        REPORTS / "notebook-protocol.json",
        {
            "declared_at_utc": datetime.now(timezone.utc).isoformat(),
            "spec_sha256": spec_hash,
            "specification": specification,
            "test_predictions_observed": False,
        },
    )
print("Preflight complete; development-only selection protocol bound to code and data.")

Preflight complete; development-only selection protocol bound to code and data.


## Training tiga kandidat dan guard sumber

TF-IDF/scaler baru di-fit pada fit gabungan. Matriks teks sementara dilepas sebelum solver bekerja agar puncak RAM lebih rendah. Kandidat dinilai pada tune saja. Bila semuanya gagal guard sumber, keputusan tetap mempertahankan frozen v6 dan artifact baru diberi status diagnostic challenger.

In [4]:
if not REUSE_FROZEN:
    with threadpool_limits(limits=4):
        raw_v6_tune = raw_probabilities({**base, "kind": "repeat_v6"}, tune.core)
        raw_v7_tune = raw_probabilities(v7_bundle, tune.core)
    reference = selection_diagnostics(tune, raw_v6_tune)
    control_validation = {
        "v6": reference,
        "v7": selection_diagnostics(tune, raw_v7_tune),
    }
    write_json(REPORTS / "frozen-control-validation.json", control_validation)
    print("Preparing expanded fit features and refitting vocabulary...", flush=True)
    fit_blocks, tune_blocks = structural_values(fit.core), structural_values(tune.core)
    assert np.array_equal(fit_blocks[0][:64], old_numeric(fit.core.iloc[:64]))
    encoder = fit_text_encoder(
        fit.core, fit.training_source, fit_blocks[0], partition="fit"
    )
    text_cache = LOCAL / "text-fit-v8.joblib"
    joblib.dump(text_matrix(encoder, fit.core), text_cache, compress=0)
    text_tune = text_matrix(encoder, tune.core)
    counts = fit.training_source.value_counts()
    validation_results = []
    for config in CONFIGS:
        print("Fit", config["name"], len(fit), "URLs...", flush=True)
        structure = None
        text_fit = joblib.load(text_cache, mmap_mode="r")
        if config["kind"] == "repeat_v6":
            numeric_fit = 0.2 * csr_matrix(
                normalize(np.clip(encoder["scaler"].transform(fit_blocks[0]), 0, 10))
            )
            numeric_tune = 0.2 * csr_matrix(
                normalize(np.clip(encoder["scaler"].transform(tune_blocks[0]), 0, 10))
            )
        else:
            structure = StructuralEncoder("linear").fit(fit_blocks, partition="fit")
            numeric_fit, numeric_tune = (
                structure.transform(fit_blocks),
                structure.transform(tune_blocks),
            )
        matrix_fit = hstack([text_fit, numeric_fit], format="csr", dtype=np.float64)
        del text_fit, numeric_fit
        gc.collect()
        weights = (
            counts.max() / fit.training_source.map(counts).to_numpy(dtype=float)
        ) ** config["source_power"]
        weights /= weights.mean()
        classifier = LogisticRegression(
            C=config["C"],
            solver="liblinear",
            max_iter=1500,
            tol=0.0001,
            random_state=SEED,
        )
        started = time.perf_counter()
        with (
            threadpool_limits(limits=4),
            warnings.catch_warnings(record=True) as caught,
        ):
            warnings.simplefilter("always", ConvergenceWarning)
            classifier.fit(matrix_fit, fit.label.to_numpy(), sample_weight=weights)
        fit_seconds = time.perf_counter() - started
        converged = not any(isinstance(w.message, ConvergenceWarning) for w in caught)
        assert converged
        features = matrix_fit.shape[1]
        del matrix_fit
        gc.collect()
        matrix_tune = hstack([text_tune, numeric_tune], format="csr", dtype=np.float64)
        raw = classifier.predict_proba(matrix_tune)[:, 1]
        candidate = {
            "encoder": encoder,
            "kind": config["kind"],
            "structure": structure,
            "classifier": classifier,
        }
        candidate_path = ARTIFACTS / (config["name"] + ".joblib")
        joblib.dump(candidate, candidate_path, compress=3)
        result = {
            **config,
            "fit_rows": len(fit),
            "features": features,
            "classifier_fit_called": True,
            "converged": converged,
            "iterations": int(classifier.n_iter_[0]),
            "fit_seconds": fit_seconds,
            "artifact_sha256": checksum(candidate_path),
            **guarded_diagnostics(tune, raw, reference),
        }
        validation_results.append(result)
        write_json(REPORTS / "validation-results.json", validation_results)
        print(
            "Completed",
            config["name"],
            "guard:",
            result["source_guard_passed"],
            "worst-first:",
            result["worst_first_score"],
            flush=True,
        )
        del candidate, classifier, matrix_tune, numeric_tune
        gc.collect()
    selected, selection_decision = choose_candidate(validation_results)
    bundle = joblib.load(ARTIFACTS / (selected["name"] + ".joblib"))
    source_guard_intervals = {}
    with threadpool_limits(limits=4):
        challenger_raw_tune = raw_probabilities(bundle, tune.core)
    for source_name, indices in tune.groupby("training_source").groups.items():
        positions = list(indices)
        source_guard_intervals[source_name] = paired_domain_interval(
            tune.iloc[positions].reset_index(drop=True),
            raw_v6_tune[positions],
            challenger_raw_tune[positions],
            reference["point"]["threshold"],
            selected["point"]["threshold"],
            repeats=1000,
        )
    write_json(REPORTS / "source-guard-domain-intervals.json", source_guard_intervals)
    write_json(
        REPORTS / "selected-before-test.json",
        {
            "selected_at_utc": datetime.now(timezone.utc).isoformat(),
            "selected": selected,
            "selection_decision": selection_decision,
            "spec_sha256": spec_hash,
            "test_predictions_observed": False,
        },
    )
    del text_tune, fit_blocks, tune_blocks
    gc.collect()
    text_cache.unlink()
else:
    control_validation = json.loads(
        (REPORTS / "frozen-control-validation.json").read_text()
    )
display(
    pd.DataFrame(
        [
            {
                k: r[k]
                for k in [
                    "name",
                    "fit_rows",
                    "macro_source_recall",
                    "min_source_recall",
                    "source_guard_passed",
                    "fit_seconds",
                ]
            }
            for r in validation_results
        ]
    )
)
print("Development decision:", selection_decision)

,name,fit_rows,macro_source_recall,min_source_recall,source_guard_passed,fit_seconds
0,expanded_rich_c2_sqrt,535504,0.455929,0.383378,False,44.704175
1,expanded_structural_c2_sqrt,535504,0.468930,0.359249,False,95.678535
2,expanded_structural_c2_equal,535504,0.477561,0.410127,False,99.247421


Development decision: {'eligible_candidates': [], 'development_selection': 'retain_frozen_v6', 'diagnostic_challenger': 'expanded_structural_c2_equal', 'challenger_passed_source_guard': False, 'production_promotion': False}


## Kalibrasi, kontrol kebijakan dan pembekuan sebelum test

Kalibrasi sigmoid hanya memakai calibration gabungan; warning/high memakai threshold partition. Aturan confidence per sumber 1%/0,1% tetap sama dengan v7, dengan fallback ukuran sampel diungkapkan. Kontrol v6 dengan kebijakan development yang sama dibekukan sebelum test. Diagnostic challenger yang gagal guard tidak dipromosikan.

In [5]:
if not REUSE_FROZEN:
    with threadpool_limits(limits=4):
        raw_calibration = np.clip(
            raw_probabilities(bundle, calibration.core), 1e-7, 1 - 1e-7
        )
        calibrator = LogisticRegression(
            C=1000, solver="lbfgs", max_iter=500, random_state=SEED
        )
        calibrator.fit(logit(raw_calibration).reshape(-1, 1), calibration.label)
        assert calibrator.coef_[0, 0] > 0
        bundle["calibrator"] = calibrator
        threshold_probability = probabilities(bundle, threshold_data.core)
        old_threshold_probability = old_probabilities(base, threshold_data.core)
    old_control = {
        "warning": source_point(
            threshold_data, old_threshold_probability, 0.01, confidence=True
        ),
        "high": source_point(
            threshold_data, old_threshold_probability, 0.001, confidence=True
        ),
    }
    warning = source_point(threshold_data, threshold_probability, 0.01, confidence=True)
    high = source_point(threshold_data, threshold_probability, 0.001, confidence=True)
    thresholds = {
        "warning": warning["threshold"],
        "high": max(high["threshold"], warning["threshold"]),
    }
    metadata = {
        "version": "notebook-url-core-v8",
        "feature_schema": SCHEMA_VERSION,
        "config": selected,
        "feature_specification": SPECIFICATION,
        "spec_sha256": spec_hash,
        "code_sha256": code_hash,
        "source_hashes": source_hashes,
        "versions": specification["versions"],
        "selection_decision": selection_decision,
        "thresholds": thresholds,
        "high_enabled": thresholds["high"] <= 1,
        "fit_rows": len(fit),
        "development_sha256": outputs["development"]["sha256"],
        "parent_v7_fit_sha256": checksum(parent_fit_path),
        "new_test_sha256": outputs["new_tr_op_test"]["sha256"],
        "threshold_policy": {"warning": warning, "high": high},
        "v6_control_thresholds": {
            level: point["threshold"] for level, point in old_control.items()
        },
        "v6_control_policy": old_control,
        "noncommercial_research_only": True,
        "production_ready": False,
        "calibration_brier_in_sample": float(
            brier_score_loss(
                calibration.label,
                calibrator.predict_proba(logit(raw_calibration).reshape(-1, 1))[:, 1],
            )
        ),
        "calibration_brier_scope": "descriptive_in_sample_not_test",
    }
    bundle["metadata"] = metadata
    joblib.dump(bundle, MODEL, compress=3)
    frozen = {
        "frozen_at_utc": datetime.now(timezone.utc).isoformat(),
        "spec_sha256": spec_hash,
        "code_sha256": code_hash,
        "source_hashes": source_hashes,
        "selected": selected,
        "selection_decision": selection_decision,
        "source_manifest_sha256": checksum(REPORTS / "frozen-data-test-manifest.json"),
        "bundle_sha256": checksum(MODEL),
        "metadata": metadata,
        "test_predictions_observed": False,
    }
    write_json(FROZEN, frozen)
assert checksum(MODEL) == frozen["bundle_sha256"]
assert bundle["metadata"] == frozen["metadata"]
assert notebook_code_hash() == frozen["code_sha256"]
reloaded = joblib.load(MODEL)
probes = [
    "http://example.org/login",
    "https://example.org/login",
    "https://alice.github.io/account",
    "https://example.org/token/REDACTED",
]
with threadpool_limits(limits=4):
    probe_probabilities = probabilities(bundle, probes)
    assert np.array_equal(probe_probabilities, probabilities(reloaded, probes))
    assert probe_probabilities[0] == probe_probabilities[1]
    assert np.isfinite(probe_probabilities).all()
artifact_checks = {
    "save_reload_exact_parity": True,
    "http_https_parity": True,
    "url_qr_same_core_pipeline": True,
    "finite_probability": True,
    "fit_only_transformations": True,
    "thresholds_selected_without_test": True,
    "protected_v3_v6_artifacts_unchanged": all(
        checksum(ROOT / p) == digest for p, digest in protected.items()
    ),
}
assert all(artifact_checks.values())
write_json(REPORTS / "artifact-checks.json", artifact_checks)
print("Frozen model:", bundle["metadata"]["version"], "selected:", selected["name"])
print("Thresholds:", bundle["metadata"]["thresholds"])

Frozen model: notebook-url-core-v8 selected: expanded_structural_c2_equal
Thresholds: {'warning': 0.9103663235299849, 'high': 0.9888354271812084}


## Holdout TR-OP baru dan benchmark teramati

Holdout baru memiliki domain berbeda dari development, tetapi sumber/penerbitnya sama dengan data tambahan training. Phish360 dan benchmark lama tetap teramati dan tidak dipakai memilih kandidat v8. Laporkan seluruh regresi, guard selection dan perubahan threshold secara eksplisit.

In [6]:
assert checksum(MODEL) == frozen["bundle_sha256"]
assert notebook_code_hash() == frozen["code_sha256"]
assert checksum(DATA / "tr-op-test-v8.parquet") == frozen["metadata"]["new_test_sha256"]
historical_path = (
    ROOT / "ml/runs/notebook-tfidf-logistic-v3/data/development-v3.parquet"
)
historical = pd.read_parquet(historical_path)
datasets = {
    "new_tr_op_domain_holdout": external,
    "observed_phish360_author_curated": pd.read_parquet(
        V7 / "data/phish360-new.parquet"
    ),
    "observed_v6_deepurlbench_holdout": pd.read_parquet(
        V6 / "data/deepurlbench-holdout-v6.parquet"
    ),
    "observed_v5_human_audit": pd.read_parquet(V5 / "data/phishvn-human-new.parquet"),
    "observed_phresh_novel_domain": historical[
        historical.partition.eq("test") & historical.novel_domain.eq(True)
    ].reset_index(drop=True),
    **{
        "observed_group_holdout_" + source: data.reset_index(drop=True)
        for source, data in historical[
            historical.partition.eq("observed_group_holdout")
        ].groupby("training_source")
    },
    "observed_urlphish": pd.read_parquet(
        ROOT / "ml/runs/notebook-tfidf-logistic-v3/data/urlphish-fresh.parquet"
    ),
    "observed_phishtank_2026_positive": pd.read_parquet(
        V6 / "data/phishtank-2026-v6.parquet"
    ),
}
evaluation, intervals, table = {}, {}, []
policies = {
    "v6_original": base["metadata"]["thresholds"],
    "v7_original": v7_bundle["metadata"]["thresholds"],
    "v6_same_confidence_policy": bundle["metadata"]["v6_control_thresholds"],
    "v8": bundle["metadata"]["thresholds"],
}
with threadpool_limits(limits=4):
    for name, data in datasets.items():
        assert set(data.group).isdisjoint(development.group)
        assert set(data.digest).isdisjoint(development.digest)
        print("Evaluate", name, len(data), flush=True)
        old_p = old_probabilities(base, data.core)
        new_p = probabilities(bundle, data.core)
        v7_p = probabilities(v7_bundle, data.core)
        evaluation[name] = {
            "rows": len(data),
            "groups": int(data.group.nunique()),
            "new_domain_holdout_same_added_source": name.startswith("new_tr_op"),
            "data_sha256": checksum(DATA / "tr-op-test-v8.parquet")
            if name.startswith("new_tr_op")
            else None,
            "models": {},
        }
        for model_name, policy in policies.items():
            p = (
                new_p
                if model_name == "v8"
                else v7_p
                if model_name == "v7_original"
                else old_p
            )
            evaluation[name]["models"][model_name] = {}
            for level, threshold in policy.items():
                item = safe_metrics(data.label, p, threshold)
                evaluation[name]["models"][model_name][level] = item
                table.append(
                    {
                        "dataset": name,
                        "model": model_name,
                        "policy": level,
                        "rows": len(data),
                        "groups": int(data.group.nunique()),
                        **{
                            key + "_pct": None if item[key] is None else 100 * item[key]
                            for key in ["accuracy", "recall", "fpr", "precision"]
                        },
                    }
                )
        intervals[name] = {
            reference: paired_domain_interval(
                data,
                v7_p if reference == "v7_original" else old_p,
                new_p,
                policies[reference]["warning"],
                policies["v8"]["warning"],
                repeats=1000,
            )
            for reference in ["v6_original", "v7_original", "v6_same_confidence_policy"]
        }
evaluation_report = {
    "evaluated_at_utc": datetime.now(timezone.utc).isoformat(),
    "bundle_sha256": frozen["bundle_sha256"],
    "code_sha256": code_hash,
    "spec_sha256": spec_hash,
    "selection_decision": selection_decision,
    "manifest_sha256": checksum(REPORTS / "frozen-data-test-manifest.json"),
    "new_test_used_for_selection_fitting_calibration_threshold": False,
    "benchmark_results": evaluation,
    "paired_domain_intervals": intervals,
}
write_json(REPORTS / "evaluation-v8.json", evaluation_report)
results_table = pd.DataFrame(table)
display(results_table[results_table.policy.eq("warning")].round(4))
print("Positive-only FPR/accuracy/traffic precision are unavailable, not zero.")

Evaluate new_tr_op_domain_holdout 1097


Evaluate observed_phish360_author_curated 7516


Evaluate observed_v6_deepurlbench_holdout 20000


Evaluate observed_v5_human_audit 33


Evaluate observed_phresh_novel_domain 80052


Evaluate observed_group_holdout_mendeley2020 1002


... (2 outputs left out)

Evaluate observed_urlphish 85824


Evaluate observed_phishtank_2026_positive 9269


,dataset,model,policy,rows,groups,accuracy_pct,recall_pct,fpr_pct,precision_pct
0,new_tr_op_domain_holdout,v6_original,warning,1097,1036,64.3573,45.3015,0.2604,99.6914
2,new_tr_op_domain_holdout,v7_original,warning,1097,1036,63.9927,44.6003,0.0000,100.0000
4,new_tr_op_domain_holdout,v6_same_confidence_policy,warning,1097,1036,60.4376,39.1304,0.0000,100.0000
6,new_tr_op_domain_holdout,v8,warning,1097,1036,73.2908,58.9060,0.0000,100.0000
8,observed_phish360_author_curated,v6_original,warning,7516,7002,75.6519,42.9866,0.1159,99.6379
10,observed_phish360_author_curated,v7_original,warning,7516,7002,75.1464,41.8932,0.1854,99.4070
12,observed_phish360_author_curated,v6_same_confidence_policy,warning,7516,7002,72.9377,36.6136,0.1159,99.5752
14,observed_phish360_author_curated,v8,warning,7516,7002,78.0601,48.8910,0.3013,99.1762
16,observed_v6_deepurlbench_holdout,v6_original,warning,20000,19139,93.2250,46.3139,0.9444,85.9060
18,observed_v6_deepurlbench_holdout,v7_original,warning,20000,19139,93.0950,43.1027,0.6914,88.5688


Positive-only FPR/accuracy/traffic precision are unavailable, not zero.


## Laporan akhir, integritas dan keputusan

Hasil challenger v8 ditampilkan sekalipun guard development gagal. Keputusan retain_v6 tidak diganti berdasarkan test. Statistik baru tidak menjadi jaminan kinerja Indonesia atau probabilitas trafik. Tidak ada label manusia baru yang dibuat agent.

In [7]:
with threadpool_limits(limits=4):
    new_tune = probabilities(bundle, tune.core)
    old_tune = old_probabilities(base, tune.core)
from guardian_ml.preprocessing import EXTRACT

parsed_tune = [prepare_url(core) for core in tune.core]
slices = {
    "private_hosting": np.array([EXTRACT(p.host).is_private for p in parsed_tune]),
    "punycode": np.array(["xn--" in p.host for p in parsed_tune]),
    "url_over_120_chars": np.array(
        [len(p.core.split("://", 1)[1]) > 120 for p in parsed_tune]
    ),
}
slice_report = {
    name: {
        "rows": int(mask.sum()),
        "v6_original": safe_metrics(
            tune.label[mask], old_tune[mask], policies["v6_original"]["warning"]
        ),
        "v8_challenger": safe_metrics(
            tune.label[mask], new_tune[mask], policies["v8"]["warning"]
        ),
    }
    for name, mask in slices.items()
}
new_test = evaluation["new_tr_op_domain_holdout"]["models"]["v8"]["warning"]
quality = {
    **selection_decision,
    "indonesia_human_gold_available": False,
    "production_ready": False,
    "new_test_recall_lower95_at_least_85pct": new_test[
        "recall_wilson95_rows_iid_assumption"
    ][0]
    >= 0.85,
    "new_test_fpr_upper95_at_most_1pct": new_test["fpr_wilson95_rows_iid_assumption"][1]
    <= 0.01,
    "new_test_is_independent_publisher": False,
    "new_test_domains_disjoint": True,
    "runtime_api_remains_v6_shadow": True,
}
assert all(checksum(ROOT / path) == digest for path, digest in protected.items())
assert notebook_code_hash() == frozen["code_sha256"]
assert checksum(MODEL) == frozen["bundle_sha256"]
summary = {
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "protocol": "ai-agent-docs/data-expansion-v8-plan.md",
    "notebook": NOTEBOOK.relative_to(ROOT).as_posix(),
    "experiment": "notebook-tfidf-logistic-v8",
    "classifier": "LogisticRegression",
    "development_audit": fit_audit,
    "feature_specification": SPECIFICATION,
    "validation_results": validation_results,
    "source_guard_domain_intervals": json.loads(
        (REPORTS / "source-guard-domain-intervals.json").read_text()
    ),
    "frozen_control_validation": control_validation,
    "selected_challenger_before_test": selected,
    "selection_decision": selection_decision,
    "frozen_at_utc": frozen["frozen_at_utc"],
    "bundle_sha256": frozen["bundle_sha256"],
    "code_sha256": code_hash,
    "spec_sha256": spec_hash,
    "thresholds": bundle["metadata"]["thresholds"],
    "threshold_policy": bundle["metadata"]["threshold_policy"],
    "new_source_audit": source_manifest,
    **evaluation_report,
    "development_structural_slices": slice_report,
    "artifact_checks": artifact_checks,
    "quality_status": quality,
    "protected_baseline_checksums": protected,
    "noncommercial_research_only": True,
    "limitations": [
        "TR-OP manual validation is reported by authors without two independent per-case audit sheets.",
        "New test uses unseen domains from the same source as added development; not independent-source or temporal evidence.",
        "TR-OP archive metadata was inspected in v7, before text intake v8.",
        "Dataset is historical 2023 and not verified Indonesian traffic.",
        "Small source confidence fallbacks and per-source regressions remain explicit.",
        "Failed guard means diagnostic challenger, not a selected replacement or production promotion.",
        "No test-based retuning, fabricated labels, target visits, or automatic deployment.",
    ],
}
write_json(REPORTS / "summary-v8.json", summary)
write_json(ROOT / "ml/reports/data-expansion-v8.json", summary)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
names = [
    "new_tr_op_domain_holdout",
    "observed_phish360_author_curated",
    "observed_phishtank_2026_positive",
]
for axis, metric in zip(axes, ["recall_pct", "fpr_pct"]):
    for offset, model, color in [
        (-0.18, "v7_original", "#64748b"),
        (0.18, "v8", "#0f766e"),
    ]:
        values = [
            results_table[
                (results_table.dataset == name)
                & (results_table.model == model)
                & results_table.policy.eq("warning")
            ][metric].iloc[0]
            for name in names
        ]
        axis.bar(
            np.arange(len(names)) + offset,
            [np.nan if pd.isna(v) else v for v in values],
            width=0.36,
            label=model,
            color=color,
        )
    axis.set_xticks(
        range(3),
        ["TR-OP (new domains)", "Phish360 (observed)", "PhishTank (observed)"],
        rotation=15,
    )
    axis.set_ylabel(metric.replace("_pct", "") + " (%)")
    axis.legend()
fig.tight_layout()
fig.savefig(REPORTS / "evaluation-v8.png", dpi=150)
display(fig)
plt.close(fig)
display(pd.Series(quality, dtype=object).to_frame("status"))
print("Training/evaluation complete; original selection decision preserved.")

<Figure size 1200x450 with 2 Axes>

,status
eligible_candidates,[]
development_selection,retain_frozen_v6
diagnostic_challenger,expanded_structural_c2_equal
challenger_passed_source_guard,False
production_promotion,False
indonesia_human_gold_available,False
production_ready,False
new_test_recall_lower95_at_least_85pct,False
new_test_fpr_upper95_at_most_1pct,True
new_test_is_independent_publisher,False


Training/evaluation complete; original selection decision preserved.
